# Grocery Price Elasticity Analysis (v3, full dataset)
**Data:** Dominick's Finer Foods weekly scanner data (University of Chicago Booth), full movement files for three categories, 93 stores.

**Business question:** Should a grocer apply uniform pricing across categories, or does price sensitivity vary enough to need a segmented strategy?

**What changed from v1:** (1) full files instead of partial extracts, (2) store-SKU and week fixed effects instead of weekly averages, (3) regular-price weeks only, so promotions do not inflate price sensitivity, (4) SKUs tiered by their own regular price, (5) PROFIT treated as gross margin percent (see Section 2), (6) promotion lift measured against each store-SKU's own regular-week baseline.

## Section 1: Load and clean (full files)

In [1]:
import numpy as np
import pandas as pd

FILES = {'Analgesics': 'wana.csv', 'Cereals': 'wcer.csv', 'Frozen Juices': 'wfrj.csv'}
COLS = ['STORE', 'UPC', 'WEEK', 'MOVE', 'PRICE', 'SALE', 'PROFIT', 'OK']
DTYPES = {'STORE': 'int16', 'UPC': 'int64', 'WEEK': 'int16', 'MOVE': 'int32',
          'PRICE': 'float32', 'PROFIT': 'float32', 'OK': 'int8'}

def load_clean(path):
    raw = pd.read_csv(path, usecols=COLS, dtype=DTYPES)
    n_raw = len(raw)
    df = raw[(raw['OK'] == 1) & (raw['PRICE'] > 0) & (raw['MOVE'] > 0)].reset_index(drop=True)
    del raw
    df['is_promo'] = df['SALE'].notna()
    df['SALE'] = df['SALE'].astype('category')
    return df, n_raw

def panel_elasticity(d, n_iter=15):
    """Elasticity from log(units) ~ log(price) with store-SKU and week fixed effects.
    Standard errors are clustered by SKU. Returns (elasticity, standard error)."""
    y = np.log(d['MOVE'].to_numpy(dtype='float64'))
    x = np.log(d['PRICE'].to_numpy(dtype='float64'))
    store_sku = pd.factorize(d['UPC'].to_numpy(dtype='int64') * 1000 + d['STORE'].to_numpy(dtype='int64'))[0]
    week = pd.factorize(d['WEEK'].to_numpy())[0]
    sku = pd.factorize(d['UPC'].to_numpy())[0]
    for _ in range(n_iter):                      # alternate demeaning = absorb both fixed effects
        for g in (store_sku, week):
            cnt = np.bincount(g)
            y = y - (np.bincount(g, weights=y) / cnt)[g]
            x = x - (np.bincount(g, weights=x) / cnt)[g]
    sxx = (x * x).sum()
    b = (x * y).sum() / sxx
    resid = y - b * x
    score = np.bincount(sku, weights=x * resid)
    se = np.sqrt((score ** 2).sum()) / sxx
    return b, se

cat_rows = {}
cereals = None
for name, path in FILES.items():
    df, n_raw = load_clean(path)
    cat_rows[name] = (n_raw, len(df), df['UPC'].nunique(), df['STORE'].nunique())
    print(f"{name}: {n_raw:,} raw rows -> {len(df):,} after cleaning "
          f"({df['UPC'].nunique()} SKUs, {df['STORE'].nunique()} stores)")
    if name == 'Cereals':
        cereals = df
    else:
        # category-level elasticity is computed below, then the frame is released to save memory
        b, se = panel_elasticity(df[~df['is_promo']])
        cat_rows[name] += (b, se, int((~df['is_promo']).sum()))
        del df
print()
print(f"Total raw rows: {sum(v[0] for v in cat_rows.values()):,}")
print(f"Total cleaned rows: {sum(v[1] for v in cat_rows.values()):,}")

Analgesics: 7,339,217 raw rows -> 3,040,172 after cleaning (638 SKUs, 93 stores)
Cereals: 6,602,582 raw rows -> 4,707,776 after cleaning (489 SKUs, 93 stores)
Frozen Juices: 3,156,545 raw rows -> 2,368,157 after cleaning (175 SKUs, 93 stores)

Total raw rows: 17,098,344
Total cleaned rows: 10,116,105


## Section 2: What PROFIT means
Version 1 treated `PROFIT` as cents of profit per unit. The check below shows it behaves like a **gross margin percent**: it is flat across price levels and uncorrelated with price. A cents-per-unit measure would be much larger for $5 cereals than for $2 cereals. All profit calculations below use `PROFIT / 100` as the margin rate.

In [2]:
chk = cereals.copy()
chk['price_band'] = pd.cut(chk['PRICE'], [0, 2, 2.5, 3, 3.5, 4, 5, 999])
print(chk.groupby('price_band', observed=True)['PROFIT'].agg(['mean', 'median', 'count']).round(2))
print(f"\nCorrelation of PROFIT with PRICE: {chk['PROFIT'].corr(chk['PRICE']):.3f}")
print(f"PROFIT range: {chk['PROFIT'].min():.1f} to {chk['PROFIT'].max():.1f}  |  mean: {chk['PROFIT'].mean():.1f}")
del chk

                   mean     median    count
price_band                                 
(0.0, 2.0]    17.850000  18.840000   358418
(2.0, 2.5]    18.020000  16.190001   568284
(2.5, 3.0]    17.059999  16.270000  1132750
(3.0, 3.5]    17.139999  16.549999  1296276
(3.5, 4.0]    18.010000  17.209999   854914
(4.0, 5.0]    17.020000  16.469999   475215
(5.0, 999.0]  15.280000  15.360000    21919

Correlation of PROFIT with PRICE: -0.013
PROFIT range: -99.7 to 100.0  |  mean: 17.4


## Section 3: Category-level elasticity
Log-log regression with store-SKU and week fixed effects, regular-price weeks only, standard errors clustered by SKU.

In [3]:
cat_results = []
cer_reg = cereals[~cereals['is_promo']]
b, se = panel_elasticity(cer_reg)
cat_rows['Cereals'] += (b, se, len(cer_reg))

for name, v in cat_rows.items():
    b, se, n = v[4], v[5], v[6]
    cat_results.append({'category': name, 'elasticity': round(b, 2),
                        'ci_low': round(b - 1.96 * se, 2), 'ci_high': round(b + 1.96 * se, 2),
                        'rows_used': n})
category_df = pd.DataFrame(cat_results)
print(category_df.to_string(index=False))

     category  elasticity  ci_low  ci_high  rows_used
   Analgesics       -0.75   -0.87    -0.63    2779919
      Cereals       -1.34   -1.53    -1.16    4350772
Frozen Juices       -1.08   -1.30    -0.85    1912506


## Section 4: Cereal elasticity by price tier
Each SKU is assigned to one tier based on its own median regular price, so a SKU does not move between tiers when it goes on sale.

In [4]:
cereals = cereals.copy()
sku_regular = cereals[~cereals['is_promo']].groupby('UPC')['PRICE'].median()
sku_all = cereals.groupby('UPC')['PRICE'].median()
cereals['sku_price'] = cereals['UPC'].map(sku_regular).fillna(cereals['UPC'].map(sku_all))

tier_labels = ['Budget (<$2.50)', 'Mid ($2.50-$3.50)', 'Premium ($3.50-$5)', 'Super Premium (>$5)']
cereals['price_tier'] = pd.cut(cereals['sku_price'], bins=[0, 2.5, 3.5, 5, 999], labels=tier_labels)

tier_results = []
for tier in tier_labels:
    sub = cereals[(cereals['price_tier'] == tier) & (~cereals['is_promo'])]
    if len(sub) < 5000:
        continue
    b, se = panel_elasticity(sub)
    tier_results.append({'price_tier': tier, 'elasticity': round(b, 2),
                         'ci_low': round(b - 1.96 * se, 2), 'ci_high': round(b + 1.96 * se, 2),
                         'skus': sub['UPC'].nunique(), 'rows_used': len(sub)})
tier_df = pd.DataFrame(tier_results)
print(tier_df.to_string(index=False))

         price_tier  elasticity  ci_low  ci_high  skus  rows_used
    Budget (<$2.50)       -0.96   -1.31    -0.61   120     709555
  Mid ($2.50-$3.50)       -1.63   -1.84    -1.42   265    2617247
 Premium ($3.50-$5)       -1.26   -1.69    -0.83    97    1016421
Super Premium (>$5)       -2.36   -2.53    -2.19     6       7549


## Section 5: Pricing scenarios for mid-tier cereals
This is a scenario analysis, not an optimization. Elasticity is estimated from small regular-price changes, so scenarios are limited to moves of 10% or less. Demand is modeled as Q1/Q0 = (P1/P0)^elasticity, and the gross margin rate is the average observed margin. The low and high cases use the 95% confidence bounds on elasticity.

In [5]:
mid = cereals[(cereals['price_tier'] == 'Mid ($2.50-$3.50)') & (~cereals['is_promo'])]
row = tier_df[tier_df['price_tier'] == 'Mid ($2.50-$3.50)'].iloc[0]
cases = {'low elasticity (CI upper)': row['ci_high'], 'point estimate': row['elasticity'],
         'high elasticity (CI lower)': row['ci_low']}

margin = mid['PROFIT'].mean() / 100
n_weeks = mid['WEEK'].nunique()
rev_base = (mid['PRICE'] * mid['MOVE']).sum() / n_weeks * 52
gp_base = (mid['PRICE'] * mid['MOVE'] * mid['PROFIT'] / 100).sum() / n_weeks * 52
print(f"Mid-tier regular-price baseline across {mid['STORE'].nunique()} stores, {mid['UPC'].nunique()} SKUs")
print(f"  Average price: ${mid['PRICE'].mean():.2f} | average gross margin: {margin:.1%}")
print(f"  Annual revenue: ${rev_base:,.0f} | annual gross profit: ${gp_base:,.0f}\n")

rows = []
for case, e in cases.items():
    for x in [-0.10, -0.05, 0.05, 0.10]:
        q_ratio = (1 + x) ** e
        rev_ratio = (1 + x) * q_ratio
        gp_ratio = (margin + x) / margin * q_ratio
        rows.append({'case': case, 'elasticity': e, 'price_change': f"{x:+.0%}",
                     'volume_%': round((q_ratio - 1) * 100, 1),
                     'revenue_$': round(rev_base * (rev_ratio - 1)),
                     'gross_profit_$': round(gp_base * (gp_ratio - 1))})
scenario_df = pd.DataFrame(rows)
print(scenario_df.to_string(index=False))

Mid-tier regular-price baseline across 93 stores, 265 SKUs
  Average price: $3.12 | average gross margin: 17.2%
  Annual revenue: $18,313,376 | annual gross profit: $3,059,966

                      case  elasticity price_change  volume_%  revenue_$  gross_profit_$
 low elasticity (CI upper)       -1.42         -10%      16.1     828591        -1570635
 low elasticity (CI upper)       -1.42          -5%       7.6     398809         -724752
 low elasticity (CI upper)       -1.42          +5%      -6.7    -371456          624478
 low elasticity (CI upper)       -1.42         +10%     -12.7    -718610         1165244
            point estimate       -1.63         -10%      18.7    1256841        -1537315
            point estimate       -1.63          -5%       8.7     601458         -699462
            point estimate       -1.63          +5%      -7.6    -554350          586920
            point estimate       -1.63         +10%     -14.4   -1067271         1081517
high elasticity (CI lo

## Section 6: Promotion analysis
Each promoted store-SKU-week is compared with that store-SKU's own average regular-price week. Gross profit per week is units x price x margin rate.

In [6]:
df = cereals
df = df.assign(gp_week=df['PRICE'] * df['MOVE'] * df['PROFIT'] / 100)
reg = df[~df['is_promo']].groupby(['UPC', 'STORE'], observed=True).agg(
    base_price=('PRICE', 'mean'), base_units=('MOVE', 'mean'), base_gp=('gp_week', 'mean')).reset_index()
promo = df[df['is_promo']].merge(reg, on=['UPC', 'STORE'], how='inner')

names = {'B': 'B (ad)', 'S': 'S (store coupon)', 'G': 'G (general)', 'C': 'C (checkout coupon)'}
rows = []
for t in ['B', 'S', 'G', 'C']:
    s = promo[promo['SALE'] == t]
    if len(s) == 0:
        continue
    rows.append({'type': names[t], 'obs': len(s),
                 'price_vs_regular_%': round((s['PRICE'].sum() / s['base_price'].sum() - 1) * 100, 1),
                 'units_vs_regular_%': round((s['MOVE'].sum() / s['base_units'].sum() - 1) * 100, 1),
                 'avg_margin_%': round(s['PROFIT'].mean(), 1),
                 'gross_profit_vs_regular_%': round((s['gp_week'].sum() / s['base_gp'].sum() - 1) * 100, 1)})
allp = {'type': 'All promotions', 'obs': len(promo),
        'price_vs_regular_%': round((promo['PRICE'].sum() / promo['base_price'].sum() - 1) * 100, 1),
        'units_vs_regular_%': round((promo['MOVE'].sum() / promo['base_units'].sum() - 1) * 100, 1),
        'avg_margin_%': round(promo['PROFIT'].mean(), 1),
        'gross_profit_vs_regular_%': round((promo['gp_week'].sum() / promo['base_gp'].sum() - 1) * 100, 1)}
promo_df = pd.DataFrame(rows + [allp])
print(f"Regular-price average margin: {df.loc[~df['is_promo'], 'PROFIT'].mean():.1f}%\n")
print(promo_df.to_string(index=False))

Regular-price average margin: 17.6%

               type    obs  price_vs_regular_%  units_vs_regular_%  avg_margin_%  gross_profit_vs_regular_%
             B (ad) 250375          -11.000000               148.5          16.4                       17.9
   S (store coupon)  91038          -24.900000               443.6          13.3                      115.5
        G (general)  11058          -32.400002               675.4          -2.4                     -507.0
C (checkout coupon)   3289           -3.800000               678.3          19.5                      743.4
     All promotions 355760          -15.500000               254.3          15.1                       36.7


## Section 7: Store pricing zones
Stores are grouped into quartiles by a price index: each store's average of (price / that SKU's median price), which controls for differences in product mix. Elasticity is then estimated within each zone.

In [7]:
idx = cereals.assign(rel_price=cereals['PRICE'] / cereals['sku_price'])
store_index = idx.groupby('STORE')['rel_price'].mean()
zone_of = pd.qcut(store_index, 4, labels=['Zone 1 (lowest prices)', 'Zone 2', 'Zone 3', 'Zone 4 (highest prices)'])
cereals['zone'] = cereals['STORE'].map(zone_of)

zone_rows = []
for z in zone_of.cat.categories:
    sub = cereals[(cereals['zone'] == z) & (~cereals['is_promo'])]
    b, se = panel_elasticity(sub)
    zone_rows.append({'zone': z, 'stores': sub['STORE'].nunique(),
                      'price_index': round(store_index[zone_of == z].mean(), 3),
                      'elasticity': round(b, 2), 'ci_low': round(b - 1.96 * se, 2),
                      'ci_high': round(b + 1.96 * se, 2)})
zone_df = pd.DataFrame(zone_rows)
print(zone_df.to_string(index=False))

                   zone  stores  price_index  elasticity  ci_low  ci_high
 Zone 1 (lowest prices)      24        0.964       -1.32   -1.51    -1.14
                 Zone 2      23        0.984       -1.33   -1.54    -1.13
                 Zone 3      23        0.997       -1.34   -1.53    -1.15
Zone 4 (highest prices)      23        1.027       -1.35   -1.54    -1.16


## Limitations
- Elasticities come from historical store-level price variation. Regular-price changes are small, so results should not be extrapolated to large price moves.
- Retailers may set prices in response to demand, which can bias elasticity estimates toward zero.
- Promotion comparisons use each store-SKU's regular weeks as the baseline and do not capture stockpiling, cannibalization of other products, or store traffic effects.
- Gross margin is the margin recorded in the data and excludes operating costs.
- Dominick's was a 1990s Chicago chain, so results are illustrative and should be validated on current data before use.